# Immobilier et performance énergétique en France

Projet du cours Python pour la data science, ENSAE Paris.
Marc Olivier Bouiti Banza, Karlane Vasthi Foudjin Sipeuwo, Aniela Nomwendé Tania Zoungrana.

Ce notebook est le rapport du projet. Il s'exécute du début à la fin à partir des fichiers produits par la chaîne de traitement (voir le README).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE))
from src import chargement, config

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.2f}".format)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})

ORDRE = chargement.ETIQUETTES
COULEURS = {"A": "#009C6D", "B": "#52B153", "C": "#78BD76", "D": "#F4E70F",
            "E": "#F0B40F", "F": "#EB8235", "G": "#D7221F"}

## Introduction

*À rédiger.* La question, le contexte (réforme du DPE de juillet 2021, loi Climat et résilience, gel des loyers des logements F et G, interdiction de louer les logements G depuis 2025) et l'étude de référence (Chareyron, 2026).

## Partie 1. Les données

### 1.1 Sources et traitement

*À rédiger.* Les trois sources, la chaîne de traitement et les six règles d'appariement (détail en annexe A).

![Chaîne de traitement](../docs/chaine_de_traitement.png)

In [ ]:
ventes = chargement.charger_ventes()
echantillon = chargement.echantillon_analyse(ventes)
print(f"{len(ventes):,} ventes ; {ventes['apparie'].sum():,} appariées ; "
      f"{len(echantillon):,} dans l'échantillon d'analyse")

### 1.2 Le marché

Prix médian au m² de toutes les ventes, appariées ou non, par trimestre et par département.

In [ ]:
prix_trimestre = (ventes.groupby(["trimestre", "type_bien"], observed=True)["prix_m2"]
                  .median()
                  .unstack())

ax = prix_trimestre.plot(marker="o", color={"appartement": "#14213D", "maison": "#A23B3B"})
ax.set_xticks(range(len(prix_trimestre)), prix_trimestre.index, rotation=45, ha="right")
ax.set(title="Prix médian au m² par trimestre (toutes les ventes)", xlabel="", ylabel="€/m²")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
import geopandas as gpd
from matplotlib.colors import ListedColormap

# Contours des départements (projet france-geojson de Grégoire David)
URL_DEPARTEMENTS = ("https://raw.githubusercontent.com/gregoiredavid/france-geojson/"
                    "master/departements-version-simplifiee.geojson")
departements = gpd.read_file(URL_DEPARTEMENTS)

prix_dep = ventes.groupby("code_departement", observed=True)["prix_m2"].median().reset_index()
prix_dep["code_departement"] = prix_dep["code_departement"].astype(str)
carte = departements.merge(prix_dep, left_on="code", right_on="code_departement", how="left")

# Classes de prix : sinon Paris (plus de 10 000 €/m²) écraserait toutes les autres couleurs
CLASSES = [0, 1500, 2000, 2500, 3000, 4000, 20000]
NOMS = ["moins de 1 500", "1 500 à 2 000", "2 000 à 2 500", "2 500 à 3 000", "3 000 à 4 000", "plus de 4 000"]
carte["classe_prix"] = pd.cut(carte["prix_m2"], CLASSES, labels=NOMS)

bleus = ListedColormap(plt.cm.Blues(np.linspace(0.25, 1, len(NOMS))))
fig, ax = plt.subplots(figsize=(7, 7))
carte.plot(column="classe_prix", cmap=bleus, categorical=True, legend=True, ax=ax,
           edgecolor="white", linewidth=0.3,
           missing_kwds={"color": "lightgrey", "label": "hors champ"},
           legend_kwds={"title": "€/m²", "loc": "center left", "bbox_to_anchor": (1, 0.5), "frameon": False})
ax.set_title("Prix médian au m² par département, 2022 à 2025")
ax.set_axis_off()
plt.show()

En gris : la Moselle, le Bas-Rhin et le Haut-Rhin, absents de la base des ventes (régime du livre foncier).

*À rédiger.* Commentaire des deux graphiques.

### 1.3 L'échantillon

Part des ventes auxquelles un DPE a été rattaché, puis comparaison des ventes appariées et non appariées. Le détail par département est en annexe B.

In [ ]:
taux = (ventes.groupby(["annee", "type_bien"], observed=True)["apparie"]
        .mean()
        .mul(100)
        .unstack())
taux["ensemble"] = ventes.groupby("annee")["apparie"].mean() * 100
taux.round(1)

In [ ]:
statut = ventes["apparie"].map({True: "appariées", False: "non appariées"})

comparaison = (ventes.groupby(["annee", statut], observed=True)
               .agg(ventes=("prix_m2", "size"),
                    prix_m2_median=("prix_m2", "median"),
                    surface_mediane=("surface", "median"),
                    part_maisons=("type_bien", lambda x: (x == "maison").mean() * 100))
               .round(1))
comparaison.index.names = ["année", "statut"]
comparaison

*À rédiger.* Commentaire : évolution du taux d'appariement et différences entre ventes appariées et non appariées.

## Partie 2. Les résultats

Toutes les analyses de cette partie portent sur l'échantillon d'analyse : les ventes appariées, sans les ventes rapprochées.

### 2.1 Description par étiquette

*À faire.* Tableau par étiquette (prix, surface, type de bien, période de construction), carte de la part de logements F ou G par département, matrice de corrélation de Spearman.

### 2.2 Comparaison sans correction

*À faire.* Boîtes à moustaches du prix au m² par étiquette, maisons et appartements séparés.

## 2.3. Résultats principaux

Cette partie présente les différences de prix au mètre carré observées entre les logements selon leur étiquette énergétique. Deux références sont utilisées : la classe G, qui correspond à la moins bonne performance énergétique, et la classe D, située au milieu de l'échelle.

Il faut distinguer deux mesures. Le premier écart compare directement les prix moyens des logements de deux classes. Il ne tient donc pas compte des différences de composition entre les groupes. Le second écart est calculé sur des groupes de logements comparables, constitués à partir de la commune, du type de bien, de la tranche de surface, de la période de construction et du trimestre de vente. Cette seconde mesure permet de limiter l'influence de ces caractéristiques sur la comparaison.

Les intervalles de confiance à 95 % sont obtenus par bootstrap, en rééchantillonnant les communes. Cette méthode tient compte du fait que les observations d'une même commune peuvent être liées entre elles. Les estimations présentées reposent sur 1 000 répétitions.

### 2.3.1. Comparaison avec la classe G

| Étiquette | Écart brut (%) | Écart entre logements comparables (%) | IC à 95 % |
|:--:|--:|--:|:--:|
| A | +32,38 | +41,29 | [29,36 ; 52,22] |
| B | +29,04 | +28,67 | [23,49 ; 32,81] |
| C | +23,72 | +17,28 | [15,29 ; 19,54] |
| D | +11,93 | +10,76 | [9,06 ; 12,60] |
| E | +9,36 | +6,27 | [5,32 ; 7,29] |
| F | +0,27 | +2,54 | [1,86 ; 3,22] |

Les écarts sont positifs pour toutes les classes comparées à G. Autrement dit, dans notre échantillon, les logements ayant une meilleure étiquette énergétique se vendent en moyenne plus cher que les logements classés G. L'écart est particulièrement marqué pour les classes A, B et C.

La comparaison entre logements similaires donne également des écarts positifs. Elle montre notamment que la différence entre F et G, peu visible dans la comparaison brute, atteint 2,54 % après prise en compte des caractéristiques retenues pour former les groupes comparables.

Il faut toutefois rester prudent pour les classes A et B. Elles représentent respectivement 0,42 % et 2,04 % des logements de la classe de référence utilisée pour cette comparaison. Leur faible représentation rend les estimations plus sensibles à la composition de l'échantillon.

### 2.3.2. Comparaison avec la classe D

| Étiquette | Écart brut (%) | Écart entre logements comparables (%) | IC à 95 % |
|:--:|--:|--:|:--:|
| A | +18,27 | +13,36 | [10,13 ; 16,64] |
| B | +15,29 | +10,96 | [8,89 ; 12,92] |
| C | +10,54 | +3,94 | [3,46 ; 4,42] |
| E | -2,29 | -4,21 | [-4,58 ; -3,85] |
| F | -10,41 | -8,72 | [-9,40 ; -8,10] |
| G | -10,66 | -11,00 | [-12,11 ; -10,01] |

La classe D permet d'observer les écarts de part et d'autre d'une étiquette intermédiaire. Les classes A, B et C présentent des prix plus élevés que D, tandis que les classes E, F et G présentent des prix plus faibles.

Après comparaison de logements similaires, l'écart atteint -8,72 % pour F et -11,00 % pour G par rapport à D. Les intervalles de confiance correspondants ne contiennent pas zéro. Dans le cadre de cette méthode, les écarts estimés sont donc relativement précis.

La comparaison brute et la comparaison entre logements similaires ne donnent pas toujours le même ordre de grandeur. C'est notamment le cas de la classe C, pour laquelle l'écart avec D passe de 10,54 % à 3,94 %. Cela montre l'importance de tenir compte des caractéristiques des logements et de leur localisation avant d'interpréter les différences de prix.

![Écarts de prix selon l'étiquette énergétique, avec G et D comme références](../reports/figures/ecarts_prix_references.png)

Dans l'ensemble, les résultats font apparaître une association entre la performance énergétique et le prix de vente. Ils ne permettent cependant pas d'affirmer que l'amélioration de l'étiquette énergétique entraîne, à elle seule, une hausse du prix. D'autres caractéristiques, qui ne sont pas toutes observées dans les données, peuvent également expliquer une partie des différences.



## 2.4. Évolution des écarts de prix dans le temps

Les résultats précédents portent sur l'ensemble de la période étudiée. Il est également intéressant de vérifier si les écarts entre les logements classés F ou G et ceux classés D évoluent d'une année à l'autre.

Pour cette analyse, la classe D est conservée comme référence. Les écarts sont calculés séparément pour chaque année, selon la même logique de comparaison entre logements similaires.

| Année | Classe | Écart brut (%) | Écart comparable (%) | Part de la cible couverte (%) |
|:--:|:--:|--:|--:|--:|
| 2022 | F | -9,35 | -6,72 | 43,19 |
| 2023 | F | -11,72 | -8,34 | 44,53 |
| 2024 | F | -10,81 | -9,05 | 45,13 |
| 2025 | F | -10,22 | -10,54 | 48,22 |
| 2022 | G | -9,59 | -8,07 | 35,33 |
| 2023 | G | -12,15 | -10,95 | 37,33 |
| 2024 | G | -11,20 | -11,32 | 37,92 |
| 2025 | G | -10,01 | -13,18 | 42,33 |

![Évolution annuelle des écarts de prix entre les classes F et G et la classe D](../reports/figures/evolution_annuelle_F_G.png)

Les résultats montrent que les logements classés F et G se vendent moins cher que ceux de classe D sur chacune des années observées. L'écart comparable devient globalement plus important au cours de la période.

Pour les logements classés F, il passe de -6,72 % en 2022 à -10,54 % en 2025. Pour les logements classés G, il évolue de -8,07 % à -13,18 %. Cette évolution est particulièrement visible pour la classe G.

Cette période correspond également à plusieurs changements réglementaires concernant les logements les moins performants sur le plan énergétique :

- **24 août 2022 :** entrée en vigueur du gel des augmentations de loyers pour les logements classés F et G en France métropolitaine, pour les contrats concernés.
- **1er janvier 2023 :** le seuil de 450 kWh d'énergie finale par mètre carré et par an entre dans le critère de décence énergétique. Les logements qui dépassent ce seuil ne satisfont plus ce critère.
- **1er janvier 2025 :** les logements classés G sont concernés par l'interdiction de location au titre du calendrier de décence énergétique.

Ces dates sont reportées sur le graphique afin de situer l'évolution des écarts dans leur contexte réglementaire. Elles ne permettent toutefois pas d'attribuer directement les variations observées à ces mesures. D'autres facteurs ont pu influencer le marché immobilier pendant la même période.

Enfin, la part de logements de la classe cible couverte par les comparaisons augmente au fil des années. Les résultats annuels doivent donc être interprétés en tenant compte de cette évolution de la couverture. Les intervalles de confiance présentés dans la section précédente ne sont pas recalculés ici pour chaque année.



## Partie 3. Vérification de la robustesse et limites de l'étude

### 3.1. Vérification avec un appariement plus strict

Les résultats peuvent dépendre des règles utilisées pour considérer deux logements comme comparables. Pour vérifier ce point, une seconde analyse a été réalisée avec des critères d'appariement plus stricts.

Les estimations obtenues avec cette méthode sont comparées à celles de l'analyse principale. Les intervalles de confiance sont également présentés pour les deux méthodes.

| Référence | Classe | Écart standard (%) | IC standard à 95 % | Écart strict (%) | IC strict à 95 % |
|:--:|:--:|--:|:--:|--:|:--:|
| D | A | +13,36 | [10,13 ; 16,64] | +15,78 | [12,30 ; 19,65] |
| D | B | +10,96 | [8,89 ; 12,92] | +12,87 | [10,51 ; 15,55] |
| D | C | +3,94 | [3,46 ; 4,42] | +4,22 | [3,65 ; 4,83] |
| D | E | -4,21 | [-4,58 ; -3,85] | -4,65 | [-5,05 ; -4,25] |
| D | F | -8,72 | [-9,40 ; -8,10] | -9,51 | [-10,29 ; -8,78] |
| D | G | -11,00 | [-12,11 ; -10,01] | -11,70 | [-12,91 ; -10,64] |
| G | A | +41,29 | [29,36 ; 52,22] | +48,80 | [34,94 ; 62,90] |
| G | B | +28,67 | [23,49 ; 32,81] | +32,94 | [26,91 ; 38,22] |
| G | C | +17,28 | [15,29 ; 19,54] | +20,04 | [17,70 ; 22,74] |
| G | D | +10,76 | [9,06 ; 12,60] | +12,05 | [10,07 ; 14,11] |
| G | E | +6,27 | [5,32 ; 7,29] | +6,97 | [5,75 ; 8,27] |
| G | F | +2,54 | [1,86 ; 3,22] | +2,86 | [2,15 ; 3,67] |

Les estimations obtenues avec l'appariement strict sont légèrement plus élevées en valeur absolue que celles de l'analyse principale. C'est le cas, par exemple, de l'écart entre les classes G et D, qui passe de -11,00 % à -11,70 %.

Malgré ces différences, les résultats conservent le même sens : les logements mieux classés présentent des prix plus élevés que ceux des classes énergétiques moins performantes, dans les comparaisons effectuées. Les intervalles de confiance restent également du même côté de zéro.

Cette vérification est plutôt rassurante : les principaux constats ne reposent pas uniquement sur le choix des critères d'appariement de l'analyse initiale. Il faut néanmoins noter que les critères plus stricts réduisent le nombre de logements pouvant être comparés. La couverture de l'échantillon est donc un peu plus faible.

### 3.2. Limites de l'étude

Plusieurs limites doivent être prises en compte pour interpréter les résultats.

**La qualité de l'appariement.** Le rapprochement entre les données de ventes et les diagnostics énergétiques repose sur des informations de localisation et sur les caractéristiques des biens. Malgré les contrôles effectués, certaines correspondances peuvent rester incertaines. Ce problème peut notamment se poser dans les immeubles comportant plusieurs logements à une même adresse.

**La sélection des logements observés.** L'étude porte sur des logements vendus et pour lesquels un diagnostic énergétique a pu être associé. Elle ne couvre donc pas l'ensemble du parc immobilier français. Les logements qui ne sont pas vendus pendant la période étudiée ne figurent pas dans l'échantillon.

**Les caractéristiques non observées.** La comparaison tient compte de plusieurs éléments, comme la commune, le type de bien, la surface, la période de construction et le trimestre de vente. D'autres facteurs peuvent néanmoins influencer le prix : l'état général du logement, sa localisation précise, la présence d'un extérieur, l'étage, la qualité des équipements ou encore les travaux réalisés.

**La représentation inégale des classes énergétiques.** Certaines classes, en particulier A et B dans la comparaison avec G, sont peu représentées. Les résultats qui les concernent doivent donc être lus avec davantage de prudence.

**L'interprétation des écarts.** Les méthodes utilisées permettent de comparer les prix observés entre des groupes de logements présentant des caractéristiques proches. Elles ne suffisent pas à établir un lien de causalité. Il n'est donc pas possible d'affirmer que la seule amélioration du DPE entraîne la hausse du prix de vente.

Enfin, le bootstrap par commune permet de tenir compte de la dépendance possible entre les observations d'une même commune pour le calcul des intervalles de confiance. Il ne corrige cependant pas les autres limites liées à la qualité des données ou aux caractéristiques qui ne sont pas observées.



## Conclusion

Cette étude avait pour objectif d'examiner le lien entre la performance énergétique des logements et leur prix de vente en France. Pour cela, les données de ventes immobilières ont été rapprochées des diagnostics de performance énergétique, puis les prix ont été comparés selon les différentes étiquettes du DPE.

Les résultats font apparaître des écarts de prix selon la classe énergétique. Dans la comparaison entre logements similaires, les biens classés F se vendent en moyenne 8,72 % moins cher que ceux classés D, tandis que l'écart atteint 11,00 % pour les logements classés G. Les comparaisons avec la classe G montrent également des écarts positifs pour les classes énergétiques supérieures.

L'analyse annuelle indique que les écarts entre les classes F et G et la classe D deviennent globalement plus importants entre 2022 et 2025. La vérification avec des critères d'appariement plus stricts conduit à des résultats proches de ceux de l'analyse principale. Les principaux constats semblent donc relativement stables face à cette modification de la méthode.

Ces résultats doivent toutefois être interprétés avec prudence. Les données disponibles ne permettent pas de tenir compte de toutes les caractéristiques susceptibles d'influencer le prix d'un logement. De plus, l'étude repose sur les biens vendus et correctement rapprochés avec un diagnostic énergétique. Elle met donc en évidence une association entre l'étiquette énergétique et le prix, mais ne permet pas de conclure à un effet causal du DPE sur la valeur immobilière.

Malgré ces limites, les résultats suggèrent que la performance énergétique constitue un élément à prendre en considération dans l'analyse du marché immobilier. Les écarts observés entre les classes D, F et G sont cohérents avec l'idée que les logements les moins performants peuvent faire l'objet d'une décote sur le marché. Cette question mérite une attention particulière dans un contexte où les exigences réglementaires relatives à la location des logements énergivores se renforcent.



## Annexes

### Annexe A. Règles d'appariement

*À rédiger.*

### Annexe B. Taux d'appariement

*À faire.* Tableau par année et type de bien, carte par département.

### Annexe C. Ventes appariées et non appariées

*À faire.* Comparaison détaillée, et répartition des étiquettes comparée à celle du parc de logements.

### Annexe D. Dictionnaire des variables

*À rédiger.*

## Références

*À compléter à partir de la note de projet.*